# Yelp - Анализ исходного датасета и выбор среза (EDA_1)
### Групповой проект 5  Deep Learning  

Это первый ноутбук конвейера. Полный датасет Yelp содержит около 7 млн отзывов по десяткам
городов и несколько ГБ текста. Обучать на нём все модели и хранить его целиком неудобно и
невоспроизводимо. Поэтому здесь мы:

1. смотрим на сырые данные (состав файлов, схемы, объёмы), т е что вообще есть до обработки
2. считаем по каждому городу метрики важные для обеих задач проекта
3. по прозрачным критериям выбираем срез из 2–3 городов, на котором дальше работаем

Конвейер: `01_eda_raw.ipynb` (выбор среза) -> scripts/preprocess.py (нарезка выбранных городов в parquet) -> `02_eda_slice.ipynb` (глубокий EDA готового среза под модели)

Тяжёлые файлы (review.json 5.3 гб) читаем потоково и отбирая колонки(polars), поэтому 5 гб текста в память не загружается

In [ ]:
import sys, json, math, re
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root))
from _constants import (
    RAW, BUSINESS, REVIEW, USER, TIP, FILES,
    ARTIFACTS, DEFAULT_CITIES, ENABLE_ARTIFACTS)
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titleweight"] = "bold"
STARS = [1, 2, 3, 4, 5]
SEED = 42
NB = "EDA_1"
def savefig(name):
    if not ENABLE_ARTIFACTS:
        return
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
    plt.savefig(ARTIFACTS / f"{NB}_{name}.png", bbox_inches="tight", dpi=140)
print("RAW:", RAW, "| ENABLE_ARTIFACTS:", ENABLE_ARTIFACTS)

## 1. Обзор сырых файлов

Из 5 файлов Yelp двум нашим задачам нужны 4: business(заведения), review(отзывы с
оценкой(таргет)), user(профили), tip(короткие заметки без оценки - сырьё для
inference в Задаче 2). checkin не используем. Смотрим размеры и схемы(колонки), не загружая
текст: схему берём с первых строк, объёмы потоково

In [ ]:
rows = []
for name in FILES:
    p = RAW / name
    size_mb = p.stat().st_size / 1e6 if p.exists() else 0
    schema = pl.scan_ndjson(p, infer_schema_length=500).collect_schema()
    short = name.replace("yelp_academic_dataset_", "").replace(".json", "")
    cols = ", ".join(list(schema.names())[:8])
    if len(schema) > 8:
        cols += " …"
    rows.append({"файл": short, "размер, МБ": round(size_mb), "колонок": len(schema), "колонки": cols})
files_overview = pd.DataFrame(rows)
display(files_overview)
def n_rows(p):
    q = pl.scan_ndjson(p, infer_schema_length=500)
    return q.select(pl.len()).collect(engine="streaming").item()
counts = {"business": n_rows(BUSINESS), "review": n_rows(REVIEW),
          "user": n_rows(USER), "tip": n_rows(TIP)}
for k, v in counts.items():
    print(k, ":", v, "записей")

## 2. Вспомогательные функции

normalize_city_state приводит названия городов к единому виду. Это нужно, чтобы один город не считался разными из-за написаний(Saint и St.)
norm_entropy показывает, насколько ровно распределены оценки 1-5. Чем ближе значение к 1, тем баланс лучше; чем ближе к 0, тем сильнее перекос в один класс

In [ ]:
def normalize_city_state(city, state):
    c_raw = city.fillna("").astype(str).str.strip().str.lower().str.replace(".", "", regex=False)
    c = c_raw.str.replace(r"^saint\s+", "st ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip().str.title()
    return c + ", " + state.fillna("").astype(str)
def norm_entropy(counts):
    counts = np.asarray(counts, dtype=float)
    tot = counts.sum()
    if tot == 0:
        return 0.0
    p = counts[counts > 0] / tot
    return float(-(p * np.log(p)).sum() / math.log(5))

## 3. Метрики по городам

Для выбора среза по каждому городу считаем объём отзывов, баланс оценок(norm_entropy),
долю негатива (1-2 звезды) и количество tips. Город берём из business.json, а
оценки из review.json (только колонки business_id и stars, текст не трогаем).

In [ ]:
biz = pd.read_json(BUSINESS, lines=True)
biz["city_state"] = normalize_city_state(biz["city"], biz["state"])
bid2city = dict(zip(biz["business_id"], biz["city_state"]))
rev = pl.scan_ndjson(REVIEW, infer_schema_length=1000)
rev = rev.select(["business_id", "stars"]).collect(engine="streaming")
rev = rev.with_columns(pl.col("business_id").replace_strict(bid2city, default=None).alias("city_state"))
star_ct = rev.drop_nulls("city_state").group_by(["city_state", "stars"]).len()
star_ct = star_ct.to_pandas().pivot(index="city_state", columns="stars", values="len").fillna(0)
for s in STARS:
    if s not in star_ct.columns:
        star_ct[s] = 0
star_ct = star_ct[STARS]
city_tot = star_ct.sum(axis=1).sort_values(ascending=False)
tip = pd.read_json(TIP, lines=True)[["business_id"]]
tip["city_state"] = tip["business_id"].map(bid2city)
tips_by_city = tip.dropna(subset=["city_state"]).groupby("city_state").size()

In [ ]:
per_city = {}
for cs in city_tot.index:
    counts = star_ct.loc[cs, STARS].to_numpy()
    tot = int(counts.sum())
    per_city[cs] = {
        "total_reviews": tot,
        "star_counts": {int(s): int(counts[i]) for i, s in enumerate(STARS)},
        "tips": int(tips_by_city.get(cs, 0)),
        "norm_entropy": norm_entropy(counts),
        "neg_share_1_2": float((counts[0] + counts[1]) / tot)}
tbl = pd.DataFrame(per_city).T
tbl["total_reviews"] = tbl["total_reviews"].astype(int)
tbl["tips"] = tbl["tips"].astype(int)
tbl = tbl.sort_values("total_reviews", ascending=False)
tbl = tbl[["total_reviews", "norm_entropy", "neg_share_1_2", "tips"]]
display(tbl.head(16))

In [ ]:
top = tbl.head(12)
plt.figure(figsize=(8, 5))
plt.barh(top.index[::-1], top["total_reviews"][::-1])
plt.title("Отзывов на город")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(top.index[::-1], top["norm_entropy"][::-1])
plt.title("Баланс оценок")
plt.xlim(0.7, 1.0)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(top.index[::-1], top["tips"][::-1])
plt.title("Tips на город")
plt.show()

## 4. Дополнительные проверки сырых данных (до нарезки среза)

Прежде чем фиксировать срез, полезно убедиться в общих свойствах данных, от которых зависит корректность обеих задач: глобальное распределение оценок, насколько внимание сконцентрировано в немногих городах, какого рода заведения преобладают и на каком языке отзывы

In [ ]:
g_stars = rev.group_by("stars").len().sort("stars").to_pandas()
share_sorted = city_tot / city_tot.sum()
cum = share_sorted.cumsum().reset_index(drop=True)
n_cities_80 = int((cum < 0.8).sum()) + 1

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(g_stars["stars"], g_stars["len"])
plt.title("Распределение оценок")
plt.xlabel("stars")
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(cum) + 1), cum.values)
plt.title("Доля отзывов по городам")
plt.xlabel("города по убыванию числа отзывов")
plt.xlim(0, 60)
plt.show()

In [ ]:
avg = (g_stars["stars"] * g_stars["len"]).sum() / g_stars["len"].sum()
print("Всего городов:", city_tot.shape[0])
print("80% отзывов дают городов:", n_cities_80)
print("Глобальная средняя оценка:", round(avg, 2))

In [ ]:
cats = biz["categories"].dropna().str.split(", ").explode()
top_cats = cats.value_counts().head(15)
def get_price(a):
    if isinstance(a, dict):
        v = a.get("RestaurantsPriceRange2")
        try:
            return int(v)
        except (TypeError, ValueError):
            return None
    return None
has_price = biz["attributes"].apply(get_price).notna().mean() if "attributes" in biz else 0
open_share = biz["is_open"].mean() if "is_open" in biz else float("nan")
plt.figure(figsize=(9, 5))
plt.barh(top_cats.index[::-1], top_cats.values[::-1])
plt.title("Топ-15 категорий заведений")
plt.xlabel("кол-во заведений")
plt.show()
print("Заведений с ценой:", round(has_price * 100), "%")
print("Открытых заведений:", round(open_share * 100), "%")

Основные категории про еду и услуги, что согласуется с нашими обеими задачами

In [ ]:
import langid
sample_txt = pl.scan_ndjson(REVIEW, infer_schema_length=1000)
sample_txt = sample_txt.select("text").head(4000).collect()["text"].to_list()
langs = []
for t in sample_txt:
    lang = langid.classify(str(t)[:400])[0]
    langs.append(lang)
langs = pd.Series(langs)
print("Доли языков (сэмпл 4k отзывов):")
print((langs.value_counts(normalize=True).head(5) * 100).round(2).astype(str) + " %")
en_share = (langs == "en").mean()
print("Доля английского:", round(en_share * 100), "%")

Задачу 2 будем строить на английских отзывах

## 5. Выбор городов-среза

Берём 2-3 крупных города, чтобы срез был достаточно большим, но не слишком тяжёлым

Критерии:
в каждом городе много отзывов и tips

общий объём около 600 тыс. отзывов

оценки распределены достаточно ровно

при прочих равных выбираем города с большим числом tips

Дальше перебираем комбинации городов и выбираем лучший вариант по этим критериям

In [ ]:
MIN_TOTAL, MAX_TOTAL, TARGET = 400_000, 700_000, 600_000
ENTROPY_FLOOR, MIN_TIPS_TOTAL = 0.87, 10_000
MIN_CITY_REVIEWS, MIN_CITY_TIPS = 100_000, 5_000
cand = {}
for cs, d in per_city.items():
    if d["total_reviews"] >= MIN_CITY_REVIEWS and d["tips"] >= MIN_CITY_TIPS:
        cand[cs] = d
results = []
for k in (2, 3):
    for combo in combinations(cand, k):
        pooled = np.zeros(5)
        total = tips = 0
        for cs in combo:
            pooled += np.array([cand[cs]["star_counts"][s] for s in STARS])
            total += cand[cs]["total_reviews"]
            tips += cand[cs]["tips"]
        ent = norm_entropy(pooled)
        if MIN_TOTAL <= total <= MAX_TOTAL and tips >= MIN_TIPS_TOTAL and ent >= ENTROPY_FLOOR:
            results.append({"cities": list(combo), "total_reviews": total, "tips_total": tips,
                            "pooled_norm_entropy": round(ent, 4),
                            "pooled_neg_share_1_2": round((pooled[0] + pooled[1]) / total, 4)})

In [ ]:
def sort_key(r):
    return -r["tips_total"], abs(r["total_reviews"] - TARGET)
results.sort(key=sort_key)
best = results[0]
print("Лучшие наборы (по tips, затем близости к 600k):")
for r in results[:6]:
    names = []
    for cs in r["cities"]:
        names.append(cs.split(",")[0])
    names = " + ".join(names)
    print(names, r["total_reviews"], "отзывов", r["tips_total"], "tips", "ent=", r["pooled_norm_entropy"])
print("ВЫБРАН СРЕЗ:", best["cities"])
print(best["total_reviews"], "отзывов")
print(best["tips_total"], "tips")
print("баланс:", best["pooled_norm_entropy"])
print("негатив:", round(best["pooled_neg_share_1_2"] * 100, 1), "%")
print("DEFAULT_CITIES в _constants.py:", DEFAULT_CITIES)
print("Совпадает с выбором:", sorted(best["cities"]) == sorted(DEFAULT_CITIES))

## 6. Итог

Алгоритм выбрал срез, зафиксированный в _constants.DEFAULT_CITIES

Выбранный срез городов:

Tucson, AZ

St Petersburg, FL

Edmonton, AB

В сумме это около 665 тыс отзывов и 92 тыс tips. Города достаточно разные: юг США и Канада. Это помогает не обучать модель только на одном локальном рынке

Дальше этот срез используется в scripts/preprocess.py: данные сохраняются в data/processed/*.parquet. Более подробный анализ уже делается в `02_eda_slice.ipynb`.